# Klasik Görüntü İşleme ile Mikroskobik Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

**Ders:** Görüntü İşleme — Yüksek Lisans Ara Sınav Projesi
**Veri seti:** Cell Tracking Challenge — Fluo-N2DH-GOWT1 (eğitim arşivi)
**Random seed:** `31084` (öğrenci numarası son 5 hane `29058` + `2026`)

Bu notebook baştan sona sırayla çalıştırıldığında aynı veri bölünmesini, aynı deneyleri ve aynı sayısal sonuçları üretir.

> GitHub'a yüklenen sürümde veri setine ait görüntü çıktıları temizlenmiştir (ödev Bölüm 18.5).

## 0. Kurulum ve konfigürasyon

In [ ]:
# Colab'da çalışıyorsak eksik paketleri kur
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagecodecs', 'tifffile'], check=True)

In [ ]:
import os, json, time, zipfile, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tifffile
import skimage
from scipy import ndimage as ndi

# ---- Sabitler ----
STUDENT_ID_LAST5 = 29058
SEED = STUDENT_ID_LAST5 + 2026          # = 31084
DATASET_URL = 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip'

# ---- Klasörler ----
PROJECT_DIR = Path.cwd()
DATA_DIR    = PROJECT_DIR / 'data'                      # .gitignore'da: veri GitHub'a gitmez
DATASET_DIR = DATA_DIR / 'Fluo-N2DH-GOWT1'
FIG_DIR     = PROJECT_DIR / 'figures'                   # veri görüntüsü içermeyen grafikler
PRIVATE_FIG_DIR = PROJECT_DIR / 'figures_private'       # veri görüntüsü içeren şekiller (sadece rapor)
for d in (DATA_DIR, FIG_DIR, PRIVATE_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('SEED =', SEED)
print('OpenCV', cv2.__version__, '| scikit-image', skimage.__version__, '| NumPy', np.__version__)

In [ ]:
# Veri seti yoksa resmi bağlantıdan indir ve aç
if not DATASET_DIR.exists():
    zip_path = DATA_DIR / 'Fluo-N2DH-GOWT1.zip'
    if not zip_path.exists():
        print('İndiriliyor...')
        urllib.request.urlretrieve(DATASET_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
print(sorted(p.name for p in DATASET_DIR.iterdir()))

In [ ]:
def frame_path(seq: str, t: int) -> Path:
    """Ham görüntü yolu, ör. ('01', 5) -> .../01/t005.tif"""
    return DATASET_DIR / seq / f't{t:03d}.tif'

def read_frame(seq: str, t: int) -> np.ndarray:
    return tifffile.imread(frame_path(seq, t))

def n_frames(seq: str) -> int:
    return len(list((DATASET_DIR / seq).glob('t*.tif')))

## 1. Veri setinin ön incelemesi (Ödev Bölüm 2)

Bu bölümde **sadece ham görüntüler** (her iki sekans) ve **Sequence 01'in** referans maskeleri incelenir.
Sequence 02'nin referans anotasyonlarına (`02_GT`) test kilidine kadar dokunulmaz.

### 1.1 Boyut, veri tipi ve kare sayısı

In [ ]:
rows = []
for seq in ['01', '02']:
    img = read_frame(seq, 0)
    rows.append({'Sekans': seq, 'Kare sayısı': n_frames(seq), 'Boyut': img.shape,
                 'dtype': str(img.dtype), 'Teorik aralık': f'0–{np.iinfo(img.dtype).max}'})
pd.DataFrame(rows)

### 1.2 Kare bazında yoğunluk istatistikleri

In [ ]:
stats = []
for seq in ['01', '02']:
    for t in range(n_frames(seq)):
        im = read_frame(seq, t).astype(np.float64)
        stats.append({'seq': seq, 't': t, 'mean': im.mean(), 'std': im.std(),
                      'min': im.min(), 'max': im.max(),
                      'p01': np.percentile(im, 1), 'p50': np.percentile(im, 50),
                      'p99': np.percentile(im, 99),
                      'frac_zero': (im == 0).mean()})
stats = pd.DataFrame(stats)
summary = stats.groupby('seq')[['mean', 'std', 'min', 'max', 'p01', 'p50', 'p99', 'frac_zero']].agg(['mean', 'min', 'max']).round(2)
summary.T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for seq, c in [('01', 'tab:blue'), ('02', 'tab:orange')]:
    s = stats[stats.seq == seq]
    axes[0].plot(s.t, s['mean'], c=c, label=f'Seq {seq}')
    axes[1].plot(s.t, s['p99'], c=c, label=f'Seq {seq}')
    axes[2].plot(s.t, s['max'], c=c, label=f'Seq {seq}')
for ax, title in zip(axes, ['Ortalama yoğunluk', '99. yüzdelik', 'Maksimum']):
    ax.set_title(title); ax.set_xlabel('Kare (t)'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig_intensity_over_time.png', dpi=150); plt.show()

### 1.3 Örnek kareler ve histogramlar

In [ ]:
sample_t = [0, 45, 91]
fig, axes = plt.subplots(2, 4, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 1, 1, 1.3]})
for r, seq in enumerate(['01', '02']):
    for c, t in enumerate(sample_t):
        im = read_frame(seq, t)
        axes[r, c].imshow(im, cmap='gray', vmin=0, vmax=255)
        axes[r, c].set_title(f'Seq {seq} — t{t:03d} (0–255 sabit ölçek)'); axes[r, c].axis('off')
        axes[r, 3].hist(im.ravel(), bins=256, range=(0, 256), histtype='step', log=True, label=f't{t:03d}')
    axes[r, 3].set_title(f'Seq {seq} histogram (log y)'); axes[r, 3].set_xlabel('Yoğunluk'); axes[r, 3].legend()
plt.tight_layout(); plt.savefig(PRIVATE_FIG_DIR / 'fig_samples_hist.png', dpi=120); plt.show()

### 1.4 Hücre boyutu ve temas eden hücreler (yalnızca Sequence 01 referans maskeleri)

`01_GT/SEG` klasöründe 92 karenin yalnızca 30'u için piksel düzeyinde maske vardır ve bu maskelerin çoğu **kısmidir**
(karedeki hücrelerin yalnızca bir kısmı işaretlenmiştir). Önce her SEG maskesindeki hücre sayısı aynı karenin TRA sayısıyla karşılaştırılır;
alan ve temas analizi yalnızca **tam işaretli** karelerde yapılır. Bu bilgi ileride minimum alan eşiği ve yapı elemanı boyutu için fiziksel referans olacaktır.

In [ ]:
def tra_count(seq: str, t: int) -> int:
    """Referans hücre sayısı: TRA etiket görüntüsündeki farklı pozitif kimlik sayısı (Ödev Bölüm 4)."""
    lab = tifffile.imread(DATASET_DIR / f'{seq}_GT' / 'TRA' / f'man_track{t:03d}.tif')
    return int(np.count_nonzero(np.unique(lab)))

seg_files = sorted((DATASET_DIR / '01_GT' / 'SEG').glob('man_seg*.tif'))
cov = []
for f in seg_files:
    t = int(f.stem.replace('man_seg', ''))
    n_seg = int(np.count_nonzero(np.unique(tifffile.imread(f))))
    cov.append({'t': t, 'SEG hücre': n_seg, 'TRA hücre': tra_count('01', t)})
cov = pd.DataFrame(cov)
cov['tam_işaretli'] = cov['SEG hücre'] == cov['TRA hücre']
full_t = cov.loc[cov.tam_işaretli, 't'].tolist()
print(f'SEG maskeli kare: {len(cov)} | tam işaretli: {len(full_t)} -> {full_t}')
cov.T

In [ ]:
areas, touching_rows = [], []
k = np.ones((3, 3), np.uint8)
for t in full_t:
    f = DATASET_DIR / '01_GT' / 'SEG' / f'man_seg{t:03d}.tif'
    lab = tifffile.imread(f)
    ids = [i for i in np.unique(lab) if i != 0]
    n_touch = 0
    for i in ids:
        m = (lab == i)
        areas.append(m.sum())
        ring = cv2.dilate(m.astype(np.uint8), k, iterations=2).astype(bool) & ~m
        if np.any((lab[ring] != 0) & (lab[ring] != i)):
            n_touch += 1
    touching_rows.append({'file': f.name, 'n_cells_seg': len(ids), 'n_touching': n_touch})
areas = np.array(areas)
touch = pd.DataFrame(touching_rows)
eq_diam = 2 * np.sqrt(areas / np.pi)
print(f'Analiz edilen tam işaretli kare: {len(full_t)}, hücre: {len(areas)}')
print(f'Hücre alanı (piksel): medyan={np.median(areas):.0f}, 5.yüzdelik={np.percentile(areas,5):.0f}, 95.yüzdelik={np.percentile(areas,95):.0f}, min={areas.min()}, max={areas.max()}')
print(f'Eşdeğer çap (piksel): medyan={np.median(eq_diam):.1f}, aralık={eq_diam.min():.1f}–{eq_diam.max():.1f}')
print(f'Temas eden (≤2 piksel komşu) hücre oranı: {touch.n_touching.sum() / touch.n_cells_seg.sum():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(areas, bins=40, color='tab:blue'); axes[0].set_title('Seq 01 hücre alanı dağılımı'); axes[0].set_xlabel('Alan (piksel)')
axes[1].hist(eq_diam, bins=40, color='tab:green'); axes[1].set_title('Eşdeğer çap'); axes[1].set_xlabel('Çap (piksel)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig_cell_size_seq01.png', dpi=150); plt.show()

### 1.5 Düzensiz arka plan ve hücreler arası parlaklık farkı

In [ ]:
t_ex = full_t[0]
im = read_frame('01', t_ex)
lab = tifffile.imread(DATASET_DIR / '01_GT' / 'SEG' / f'man_seg{t_ex:03d}.tif')
# Arka planı ölçmek için hücreleri (genişletilmiş referans maskesi) dışarıda bırak, 8x8 ızgarada ortalama al
bg_mask = cv2.dilate((lab > 0).astype(np.uint8), np.ones((15, 15), np.uint8)) == 0
G = 8; h, w = im.shape; bg_grid = np.zeros((G, G))
for i in range(G):
    for j in range(G):
        blk = im[i*h//G:(i+1)*h//G, j*w//G:(j+1)*w//G]; m = bg_mask[i*h//G:(i+1)*h//G, j*w//G:(j+1)*w//G]
        bg_grid[i, j] = blk[m].mean() if m.any() else np.nan
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(im, cmap='gray'); axes[0].set_title(f'Seq 01 t{t_ex:03d} (otomatik kontrast)'); axes[0].axis('off')
hm = axes[1].imshow(bg_grid, cmap='magma'); plt.colorbar(hm, ax=axes[1], fraction=.046)
axes[1].set_title('Hücre dışı arka plan ortalaması (8x8 ızgara)'); axes[1].axis('off')
print(f'Arka plan ızgara ortalaması: min={np.nanmin(bg_grid):.2f}, max={np.nanmax(bg_grid):.2f}')
if lab is not None:
    means = [im[lab == i].mean() for i in np.unique(lab) if i != 0]
    axes[2].hist(means, bins=15, color='tab:purple'); axes[2].set_title(f't{t_ex:03d}: hücre başına ortalama parlaklık'); axes[2].set_xlabel('Ortalama yoğunluk')
plt.tight_layout(); plt.savefig(PRIVATE_FIG_DIR / 'fig_background_brightness.png', dpi=120); plt.show()
if lab is not None:
    print(f'Hücre ortalama parlaklıkları: min={min(means):.1f}, max={max(means):.1f}  (oran ≈ {max(means)/max(min(means),1):.1f}x)')

## 2. Geliştirme, doğrulama ve test görüntülerinin seçilmesi (Ödev Bölüm 3)

| Grup | Adet | Kaynak | Kural |
|---|---|---|---|
| Doğrulama | 5 | Seq 01 | Önce seçilir |
| Geliştirme | 10 | Seq 01 | Her doğrulama karesinden en az 5 kare boşlukla ayrılır |
| Test | 15 | Seq 02 | 3 blok × 5 ardışık kare, bloklar arasında en az 5 kare boşluk |

**"En az 5 karelik boşluk" yorumu:** İki kare arasında en az 5 kare bulunmalıdır, yani `|t_i − t_j| ≥ 6`.
Bu en katı yorumdur. Ardışık kareler neredeyse aynı göründüğü için, gruplar arası benzerlikten kaynaklanan sızıntıyı önler.

**Deterministik tekrar prosedürü:** Tek bir `numpy.random.default_rng(SEED)` üreteci kullanılır. Koşul sağlanmazsa aynı üreteçle
yeniden çekim yapılır. Böylece kaç deneme gerekirse gereksin sonuç tekrarlanabilir. Deneme sayıları kaydedilir.

Bu seçim algoritma geliştirmeden **önce** yapılmıştır ve sonradan değiştirilmez.

In [ ]:
MIN_GAP = 6            # |t_i - t_j| >= 6  <=>  aralarında en az 5 kare
N_VAL, N_DEV = 5, 10
N_BLOCKS, BLOCK_LEN = 3, 5
MAX_TRIES = 10_000

rng = np.random.default_rng(SEED)
T01, T02 = n_frames('01'), n_frames('02')

# 1) Doğrulama: Seq 01'den rastgele 5 kare (ödev sırası: önce doğrulama)
val_t = sorted(rng.choice(T01, size=N_VAL, replace=False).tolist())

# 2) Geliştirme: doğrulamaya en az MIN_GAP uzaklıktaki karelerden rastgele 10 kare
dev_tries = 0
for dev_tries in range(1, MAX_TRIES + 1):
    candidates = [t for t in range(T01) if all(abs(t - v) >= MIN_GAP for v in val_t)]
    if len(candidates) >= N_DEV:
        dev_t = sorted(rng.choice(candidates, size=N_DEV, replace=False).tolist())
        break
    # Aday yetersizse doğrulama setini aynı üreteçle yeniden çek (deterministik tekrar)
    val_t = sorted(rng.choice(T01, size=N_VAL, replace=False).tolist())
else:
    raise RuntimeError('Geliştirme seçimi başarısız')

# 3) Test: Seq 02'den 3 başlangıç noktası, her biri 5 ardışık kare
#    Blok k = [s_k, s_k+4]; çakışmama + 5 kare boşluk => s_{k+1} - s_k >= BLOCK_LEN + 5
test_tries = 0
for test_tries in range(1, MAX_TRIES + 1):
    starts = sorted(rng.choice(T02 - BLOCK_LEN + 1, size=N_BLOCKS, replace=False).tolist())
    if all(b - a >= BLOCK_LEN + 5 for a, b in zip(starts, starts[1:])):
        break
else:
    raise RuntimeError('Test seçimi başarısız')

print('Doğrulama  (Seq01):', val_t)
print('Geliştirme (Seq01):', dev_t, f'| deneme={dev_tries}')
print('Test başlangıçları (Seq02):', starts, f'| deneme={test_tries}')

In [ ]:
# Kural kontrolleri (assert): biri bozulursa notebook durur
assert len(set(val_t) | set(dev_t)) == N_VAL + N_DEV, 'çakışma var'
assert min(abs(d - v) for d in dev_t for v in val_t) >= MIN_GAP, 'dev-val boşluğu yetersiz'
blocks = [list(range(s, s + BLOCK_LEN)) for s in starts]
for (a, b) in zip(blocks, blocks[1:]):
    assert b[0] - a[-1] - 1 >= 5, 'test blokları arası boşluk yetersiz'
assert all(0 <= t < T02 for blk in blocks for t in blk)
print('Tüm seçim kuralları sağlandı. Dev-val minimum mesafe:',
      min(abs(d - v) for d in dev_t for v in val_t))

In [ ]:
rows = []
rows += [{'group': 'validation', 'sequence': '01', 'frame': t, 'block': ''} for t in val_t]
rows += [{'group': 'development', 'sequence': '01', 'frame': t, 'block': ''} for t in dev_t]
for k, blk in enumerate(blocks, start=1):
    rows += [{'group': 'test', 'sequence': '02', 'frame': t, 'block': f'B{k}'} for t in blk]
selected = pd.DataFrame(rows)
selected['filename'] = selected.apply(lambda r: f"{r.sequence}/t{r.frame:03d}.tif", axis=1)
selected['seed'] = SEED
selected = selected[['group', 'sequence', 'block', 'frame', 'filename', 'seed']]
selected.to_csv(PROJECT_DIR / 'selected_frames.csv', index=False)
selected.groupby('group').size()

In [ ]:
# Zaman ekseninde seçimlerin görünümü (veri görüntüsü içermez)
fig, axes = plt.subplots(2, 1, figsize=(14, 2.8), sharex=True)
axes[0].scatter(val_t, [0]*N_VAL, marker='s', s=70, c='tab:red', label='Doğrulama')
axes[0].scatter(dev_t, [0]*N_DEV, marker='o', s=70, c='tab:blue', label='Geliştirme')
axes[0].set_title('Sequence 01'); axes[0].legend(loc='upper right', ncol=2)
for k, blk in enumerate(blocks, 1):
    axes[1].scatter(blk, [0]*BLOCK_LEN, marker='D', s=70, c='tab:green')
    axes[1].text(np.mean(blk), 0.02, f'B{k}', ha='center')
axes[1].set_title('Sequence 02 (Test)')
for ax in axes:
    ax.set_yticks([]); ax.set_xlim(-1, max(T01, T02)); ax.grid(axis='x', alpha=.3)
axes[1].set_xlabel('Kare numarası (t)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig_selected_frames_timeline.png', dpi=150); plt.show()

## 3. Referans hücre sayıları (Ödev Bölüm 4)

Referans sayı = `XX_GT/TRA/man_trackTTT.tif` etiket görüntüsünde **sıfır hariç farklı pozitif kimlik sayısı**.
Bölüm 1.4'te görüldüğü gibi SEG maskeleri kısmi olduğu için TRA kullanılır; TRA 92 karenin tamamını kapsar.

⚠️ **Yalnızca geliştirme ve doğrulama referansları okunur.** Test (Seq 02) referansları Bölüm 10'daki kilit commit'inden sonra okunacaktır.

In [ ]:
devval = selected[selected.group.isin(['development', 'validation'])].copy()
devval['ref_count'] = [tra_count(s, t) for s, t in zip(devval.sequence, devval.frame)]
devval.to_csv(PROJECT_DIR / 'reference_counts_devval.csv', index=False)
REF = {(r.sequence, r.frame): r.ref_count for r in devval.itertuples()}
devval[['group', 'frame', 'ref_count']].sort_values(['group', 'frame']).reset_index(drop=True)

### 3.1 Referansların görsel kontrolü

TRA işaretçileri (marker) her hücrenin içine yerleştirilmiş küçük bölgelerdir. Her işaretçinin merkezini görüntü üzerinde gösterip,
her hücrede tam bir işaretçi olup olmadığı gözle kontrol edilir. Görüntü gamma=0.5 ile parlatılmıştır; sönük hücreler ancak böyle görülür.

In [ ]:
def tra_centroids(seq, t):
    lab = tifffile.imread(DATASET_DIR / f'{seq}_GT' / 'TRA' / f'man_track{t:03d}.tif')
    ids = [i for i in np.unique(lab) if i != 0]
    return np.array(ndi.center_of_mass(lab > 0, lab, ids))

show = val_t[:2] + dev_t[:2]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 5))
for ax, t in zip(axes, show):
    im = read_frame('01', t).astype(float)
    ax.imshow((im / im.max()) ** 0.5, cmap='gray')
    cen = tra_centroids('01', t)
    ax.scatter(cen[:, 1], cen[:, 0], s=60, facecolors='none', edgecolors='red', linewidths=1.5)
    grp = 'val' if t in val_t else 'dev'
    ax.set_title(f'Seq01 t{t:03d} ({grp}) — referans={REF[("01", t)]}'); ax.axis('off')
plt.tight_layout(); plt.savefig(PRIVATE_FIG_DIR / 'fig_reference_check.png', dpi=110); plt.show()

## 4. Baseline yöntemi (Ödev Bölüm 5)

**Akış:** ham gri görüntü → global Otsu eşikleme → 8-komşuluklu bağlı bileşen etiketleme → minimum alan filtresi → sayım.

Ödevin izin verdiği tek ayar, **geliştirme verileri üzerinde belirlenen sabit bir minimum nesne alanı eşiğidir**.
Bu eşik 0–800 piksel aralığında 10 piksel adımla taranır ve geliştirme MAE'sini en düşük yapan değer seçilir
(eşitlik olursa küçük olan). Seçim kuralı, sonraki tüm deneylerle tutarlı olsun diye MAE'dir.
Baseline bu bölümden sonra **dondurulur**.

In [ ]:
def count_components(binary: np.ndarray, min_area: int = 0, connectivity: int = 8):
    """İkili maskede bağlı bileşenleri bulur; min_area'dan küçükleri atar.
    Dönüş: (sayı, etiket görüntüsü [atılanlar 0], alanlar)"""
    n, lab, st, _ = cv2.connectedComponentsWithStats(binary.astype(np.uint8), connectivity=connectivity)
    areas = st[1:, cv2.CC_STAT_AREA]
    keep = np.zeros(n, bool); keep[1:] = areas >= min_area
    remap = np.zeros(n, np.int32); remap[keep] = np.arange(1, keep.sum() + 1)
    return int(keep.sum()), remap[lab], areas[areas >= min_area]

def baseline_segment(img: np.ndarray):
    """Baseline: global Otsu. Dönüş: (ikili maske, Otsu eşiği)"""
    th, b = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return b > 0, th

def count_metrics(pred, ref):
    e = np.asarray(pred, float) - np.asarray(ref, float)
    return {'MAE': np.abs(e).mean(), 'RMSE': np.sqrt((e ** 2).mean()), 'Bias': e.mean(),
            'ExactAcc': (e == 0).mean()}

def match_stats(lab: np.ndarray, seq: str, t: int):
    """Sayımın arkasını açar: tespitler referans işaretçileriyle nasıl eşleşiyor?
    - hit_cells: en az bir tespitin örtüştüğü farklı referans hücre sayısı
    - empty: hiçbir işaretçiyle örtüşmeyen tespit (yanlış pozitif)
    - extra: aynı hücreye düşen fazladan tespitler (parçalanma)
    - merged: birden fazla hücre içeren tespit (birleşme)"""
    tra = tifffile.imread(DATASET_DIR / f'{seq}_GT' / 'TRA' / f'man_track{t:03d}.tif')
    hit, empty, merged, on_cell = set(), 0, 0, 0
    for k in range(1, lab.max() + 1):
        ids = np.unique(tra[lab == k]); ids = ids[ids > 0]
        if len(ids) == 0: empty += 1
        else:
            on_cell += 1; hit.update(ids.tolist()); merged += len(ids) > 1
    return {'hit_cells': len(hit), 'missed': int(np.count_nonzero(np.unique(tra))) - len(hit),
            'empty': empty, 'extra': on_cell - len(hit), 'merged': merged}

### 4.1 Minimum alan eşiğinin geliştirme verisinde belirlenmesi

In [ ]:
dev_frames = sorted(devval.loc[devval.group == 'development', 'frame'])
val_frames = sorted(devval.loc[devval.group == 'validation', 'frame'])

# Otsu maskeleri bir kez hesaplanır, farklı min_area değerleri bunlar üzerinde denenir
dev_masks = {t: baseline_segment(read_frame('01', t))[0] for t in dev_frames}
grid = list(range(0, 801, 10))
sweep = []
for A in grid:
    preds = [count_components(dev_masks[t], A)[0] for t in dev_frames]
    m = count_metrics(preds, [REF[('01', t)] for t in dev_frames])
    sweep.append({'min_area': A, **m})
sweep = pd.DataFrame(sweep)
BASELINE_MIN_AREA = int(sweep.loc[sweep.MAE.idxmin(), 'min_area'])   # idxmin eşitlikte ilkini (küçüğü) alır
print('Seçilen BASELINE_MIN_AREA =', BASELINE_MIN_AREA)
sweep[sweep.min_area.isin([0, 10, 20, 30, 40, 50, 100, 200, 300, 500, 800])].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sweep.min_area, sweep.MAE, label='MAE'); ax.plot(sweep.min_area, sweep.Bias, ls='--', label='Bias')
ax.axhline(0, c='k', lw=.5); ax.axvline(BASELINE_MIN_AREA, c='tab:red', ls=':', label=f'seçilen = {BASELINE_MIN_AREA} px')
ax.set_ylim(-15, 25); ax.set_xlabel('Minimum alan eşiği (piksel)'); ax.set_ylabel('Hücre')
ax.set_title('Baseline: minimum alan eşiğinin etkisi (10 geliştirme karesi)'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig_baseline_minarea_sweep.png', dpi=150); plt.show()

In [ ]:
# Baseline dondurulur: parametreler dosyaya yazılır ve bundan sonra değiştirilmez
BASELINE = {'name': 'baseline_otsu_cc', 'steps': ['gray', 'global_otsu', 'connected_components_8', 'min_area_filter'],
            'min_area': BASELINE_MIN_AREA, 'connectivity': 8,
            'min_area_selection': 'argmin development MAE over 0..800 step 10'}
(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
with open(PROJECT_DIR / 'configs' / 'baseline.json', 'w') as f:
    json.dump(BASELINE, f, indent=2)

def run_baseline(img):
    b, th = baseline_segment(img)
    n, lab, _ = count_components(b, BASELINE['min_area'], BASELINE['connectivity'])
    return n, lab, b, th
BASELINE

### 4.2 Baseline sonuçları (geliştirme + doğrulama)

In [ ]:
rows = []
for grp, frames in [('development', dev_frames), ('validation', val_frames)]:
    for t in frames:
        img = read_frame('01', t)
        t0 = time.perf_counter(); n, lab, b, th = run_baseline(img); dt = time.perf_counter() - t0
        rows.append({'group': grp, 'frame': t, 'ref': REF[('01', t)], 'pred': n, 'otsu_th': th,
                     'n_components_raw': count_components(b, 0)[0], 'time_s': dt, **match_stats(lab, '01', t)})
base_res = pd.DataFrame(rows)
base_res['error'] = base_res.pred - base_res.ref
base_res.to_csv(PROJECT_DIR / 'baseline_results_devval.csv', index=False)
base_res

In [ ]:
summ = []
for grp in ['development', 'validation']:
    d = base_res[base_res.group == grp]
    summ.append({'Küme': grp, **{k: round(v, 3) for k, v in count_metrics(d.pred, d.ref).items()},
                 'Bulunan hücre': f"{d['hit_cells'].sum()}/{d['ref'].sum()}", 'Parçalanma (fazla)': d['extra'].sum(),
                 'Boş tespit': d['empty'].sum(), 'Birleşme': d['merged'].sum(), 'Ort. süre (ms)': round(d.time_s.mean() * 1000, 1)})
pd.DataFrame(summ)

**Yorum notu:** Düşük Bias'ın sebebi algoritmanın iyi olması değil, iki zıt hatanın birbirini götürmesidir.
Kaçırılan sönük hücreler (eksik sayım) ile orta parlaklıktaki hücrelerin parçalanmasından gelen fazladan tespitler (fazla sayım) birbirini dengeler.
Bu nedenle sonraki deneylerde MAE'nin yanında eşleşme tablosu da raporlanacaktır.

### 4.3 Baseline ara çıktıları (Şekil 4 için)

In [ ]:
t = dev_frames[0]
img = read_frame('01', t); n, lab, b, th = run_baseline(img)
rng_c = np.random.default_rng(0); cols = rng_c.uniform(.3, 1, (lab.max() + 1, 3)); cols[0] = 0
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title(f'1) Ham görüntü t{t:03d}')
axes[1].imshow(b, cmap='gray'); axes[1].set_title(f'2) Otsu (eşik={th:.0f}) — {count_components(b, 0)[0]} bileşen')
axes[2].imshow(cols[lab]); axes[2].set_title(f'3) Min. alan ≥ {BASELINE_MIN_AREA}px — {n} nesne')
axes[3].imshow((img / img.max()) ** .5, cmap='gray'); axes[3].contour(lab > 0, colors='red', linewidths=.6)
cen = tra_centroids('01', t); axes[3].scatter(cen[:, 1], cen[:, 0], s=25, c='cyan', marker='x')
axes[3].set_title(f'4) Kırmızı = tespit, mavi × = referans ({REF[("01", t)]})')
for a in axes: a.axis('off')
plt.tight_layout(); plt.savefig(PRIVATE_FIG_DIR / 'fig_baseline_stages.png', dpi=110); plt.show()